In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import os
import torch

root = '/content/gdrive/MyDrive'
required = [
    f'{root}/DL_dataset_RSR25dB/{split}_P10.mat'
    for split in ('train', 'validation', 'test')
]
required += [
    f'{root}/unfolded_dl_v7_p10_extended_results/'
    f'model_P10_seed{seed}_extended.pt'
    for seed in (20261026, 20261027, 20261028)
]
required += [
    f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_{split}_P10.json'
    for split in ('validation', 'test')
]
missing = [path for path in required if not os.path.isfile(path)]
assert not missing, f'Missing Drive files: {missing}'
assert torch.cuda.is_available(), 'Connect a T4 GPU before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('P=10 data, V7 checkpoints, and GS curves found.')

Mounted at /content/gdrive
GPU: Tesla T4
P=10 data, V7 checkpoints, and GS curves found.


In [ ]:
from google.colab import files

uploaded = files.upload()

required_scripts = {
    'colab_unfolded_dl_v3_highsnr.py',
    'colab_unfolded_dl_v5_snr_aware.py',
    'colab_unfolded_dl_v6_gs_gap.py',
    'colab_unfolded_dl_v7_continue_p10.py',
}
missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'

print('All dependency scripts uploaded.')

Saving colab_unfolded_dl_v3_highsnr.py to colab_unfolded_dl_v3_highsnr.py
Saving colab_unfolded_dl_v5_snr_aware.py to colab_unfolded_dl_v5_snr_aware.py
Saving colab_unfolded_dl_v6_gs_gap.py to colab_unfolded_dl_v6_gs_gap.py
Saving colab_unfolded_dl_v7_continue_p10.py to colab_unfolded_dl_v7_continue_p10.py
All dependency scripts uploaded.


In [ ]:
%%writefile /content/colab_unfolded_dl_v8_continue_p10.py
"""Continue selected V7 P=10 models for up to 100 additional epochs.

The architecture, data, and loss stay the same. Checkpoint selection uses
the worst per-SNR validation gap to GS. Test data is loaded only after model
selection. V7 did not save optimizer state, so Adam restarts from 5e-5.
V8 saves resumable optimizer/model checkpoints after each epoch.
"""

import importlib.util
import json
import os
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader


SCRIPT_DIR = '/content'
DATA_ROOT = '/content/gdrive/MyDrive/DL_dataset_RSR25dB'
V7_ROOT = '/content/gdrive/MyDrive/unfolded_dl_v7_p10_extended_results'
GS_ROOT = '/content/gdrive/MyDrive/DL_dataset_RSR25dB/v6_starting_models'
OUTPUT_ROOT = '/content/gdrive/MyDrive/unfolded_dl_v8_continue_p10_results'

P = 10
SEEDS = (20261026, 20261027, 20261028)
SNR_GRID = (-5, 0, 5, 10, 15, 20, 25, 30)
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 25
INITIAL_LR = 5e-5
LR_PATIENCE = 8
MIN_LR = 1e-6
BATCH_SIZE = 128


def import_sibling(module_name, filename):
    path = os.path.join(SCRIPT_DIR, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Missing uploaded script: {path}')
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


v7 = import_sibling('unfolded_v7', 'colab_unfolded_dl_v7_continue_p10.py')
v6 = v7.v6
base = v7.base


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_gs_curve(split):
    path = os.path.join(GS_ROOT, f'gs_{split}_P10.json')
    with open(path, 'r', encoding='utf-8') as f:
        curve = json.load(f)

    data_path = os.path.join(DATA_ROOT, f'{split}_P10.mat')
    if v6.sha256_file(data_path) != curve.get('dataset_sha256'):
        raise ValueError(f'GS {split} curve does not match the dataset.')
    return curve


def state_on_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def get_validation_gap(model, loader, gs_validation, device):
    metrics = base.evaluate(model, loader, device)
    gaps = {
        str(snr): float(
            metrics['nmse_db_by_snr'][str(snr)]
            - gs_validation['nmse_db_by_snr'][str(snr)]
        )
        for snr in SNR_GRID
    }
    return metrics, gaps, max(gaps.values())


def run_seed(seed, train_data, validation_data, channel_scale,
             gs_validation, device):
    set_seed(seed + 8000)

    generator = torch.Generator()
    generator.manual_seed(seed + 8000)

    train_loader = DataLoader(
        base.ChannelDataset(train_data),
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=(device.type == 'cuda'),
        generator=generator,
    )
    validation_loader = DataLoader(
        base.ChannelDataset(validation_data),
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=(device.type == 'cuda'),
    )

    source_path = os.path.join(
        V7_ROOT, f'model_P10_seed{seed}_extended.pt'
    )
    source = torch.load(source_path, map_location='cpu', weights_only=False)

    if not np.isclose(
        float(source['channel_scale']), channel_scale, rtol=1e-6, atol=0.0
    ):
        raise ValueError('V7 checkpoint channel scale differs from training data.')

    model = base.UnfoldedMagnitudeEstimator(40).to(device)
    model.load_state_dict(source['model_state_dict'], strict=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=LR_PATIENCE,
        min_lr=MIN_LR,
    )

    _, best_gaps, best_score = get_validation_gap(
        model, validation_loader, gs_validation, device
    )
    best_state = state_on_cpu(model)
    best_epoch = 0
    history = [{
        'epoch': 0,
        'validation_dl_minus_gs_db_by_snr': best_gaps,
        'worst_validation_gap_to_gs_db': best_score,
        'note': 'Starting V7 checkpoint; Adam optimizer restarted.',
    }]
    no_improvement = 0

    resume_path = os.path.join(OUTPUT_ROOT, f'resume_P10_seed{seed}.pt')
    final_path = os.path.join(OUTPUT_ROOT, f'model_P10_seed{seed}_v8.pt')
    start_epoch = 1

    if os.path.isfile(resume_path):
        saved = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(saved['model_state_dict'])
        optimizer.load_state_dict(saved['optimizer_state_dict'])
        scheduler.load_state_dict(saved['scheduler_state_dict'])
        best_score = saved['best_score']
        best_state = saved['best_state']
        best_epoch = saved['best_epoch']
        best_gaps = saved['best_gaps']
        history = saved['history']
        no_improvement = saved['no_improvement']
        start_epoch = saved['epoch'] + 1
        print(f'Resuming seed {seed} from epoch {start_epoch}.')

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            batch = base.move_batch_to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)

            pred_real, pred_imag = model(batch)
            loss = base.weighted_nmse_loss(
                pred_real,
                pred_imag,
                batch['target_real'],
                batch['target_imag'],
                batch['snr'],
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        validation, gaps, score = get_validation_gap(
            model, validation_loader, gs_validation, device
        )
        scheduler.step(score)

        history.append({
            'epoch': epoch,
            'train_loss': total_loss / max(len(train_loader), 1),
            'learning_rate': optimizer.param_groups[0]['lr'],
            'validation_nmse_db_by_snr': validation['nmse_db_by_snr'],
            'validation_dl_minus_gs_db_by_snr': gaps,
            'worst_validation_gap_to_gs_db': score,
        })

        if score < best_score:
            best_score = score
            best_epoch = epoch
            best_gaps = gaps
            best_state = state_on_cpu(model)
            no_improvement = 0
        else:
            no_improvement += 1

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_score': best_score,
            'best_state': best_state,
            'best_epoch': best_epoch,
            'best_gaps': best_gaps,
            'no_improvement': no_improvement,
            'history': history,
        }, resume_path)

        print(
            f'P=10 seed={seed} epoch={epoch:03d} '
            f'worst validation DL-GS gap={score:.4f} dB '
            f'lr={optimizer.param_groups[0]["lr"]:.2e}',
            flush=True,
        )

        if no_improvement >= EARLY_STOPPING_PATIENCE:
            print(f'Early stopping seed {seed} at epoch {epoch}.')
            break

    # Test data is only loaded after selecting the validation checkpoint.
    model.load_state_dict(best_state)
    test_source = base.load_split(DATA_ROOT, P, 'test')
    test_data = base.prepare_split(test_source, channel_scale)
    test_loader = DataLoader(
        base.ChannelDataset(test_data),
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=(device.type == 'cuda'),
    )
    test_metrics = base.evaluate(model, test_loader, device)
    gs_test = load_gs_curve('test')

    test_gaps = {
        str(snr): float(
            test_metrics['nmse_db_by_snr'][str(snr)]
            - gs_test['nmse_db_by_snr'][str(snr)]
        )
        for snr in SNR_GRID
    }

    result = {
        'P': P,
        'seed': seed,
        'source_v7_model': source_path,
        'optimizer_restarted': True,
        'best_v8_epoch': best_epoch,
        'best_worst_validation_gap_to_gs_db': best_score,
        'selected_validation_dl_minus_gs_db_by_snr': best_gaps,
        'test_metrics': test_metrics,
        'test_dl_minus_gs_db_by_snr': test_gaps,
        'history': history,
    }

    torch.save({
        'model_state_dict': model.state_dict(),
        'channel_scale': channel_scale,
        'number_of_layers': 40,
        'result': result,
    }, final_path)

    return result


def main():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device:', device)
    if device.type == 'cuda':
        print('GPU:', torch.cuda.get_device_name(0))

    train_source = base.load_split(DATA_ROOT, P, 'train')
    channel_scale = float(
        np.sqrt(np.mean(np.abs(train_source['G_data']) ** 2))
    )
    channel_scale = max(channel_scale, np.sqrt(np.finfo(np.float32).eps))

    train_data = base.prepare_split(train_source, channel_scale)
    validation_source = base.load_split(DATA_ROOT, P, 'validation')
    validation_data = base.prepare_split(validation_source, channel_scale)
    gs_validation = load_gs_curve('validation')

    summary_path = os.path.join(
        OUTPUT_ROOT, 'summary_unfolded_v8_p10_continuation.json'
    )
    results = []

    for seed in SEEDS:
        final_path = os.path.join(OUTPUT_ROOT, f'model_P10_seed{seed}_v8.pt')

        if os.path.isfile(final_path):
            saved = torch.load(final_path, map_location='cpu', weights_only=False)
            result = saved['result']
            print(f'Using completed V8 result for seed {seed}.')
        else:
            started = time.time()
            result = run_seed(
                seed,
                train_data,
                validation_data,
                channel_scale,
                gs_validation,
                device,
            )
            result['seconds'] = time.time() - started

        results.append(result)
        with open(summary_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2)

        print(f"Seed {seed} test DL-GS gaps:",
              result['test_dl_minus_gs_db_by_snr'])

    print('Results saved to:', OUTPUT_ROOT)


if __name__ == '__main__':
    main()

Writing /content/colab_unfolded_dl_v8_continue_p10.py


In [ ]:
assert torch.cuda.is_available(), 'Connect a T4 GPU runtime first.'
!python /content/colab_unfolded_dl_v8_continue_p10.py

Device: cuda
GPU: Tesla T4
P=10 seed=20261026 epoch=001 worst validation DL-GS gap=0.8305 dB lr=5.00e-05
P=10 seed=20261026 epoch=002 worst validation DL-GS gap=0.8279 dB lr=5.00e-05
P=10 seed=20261026 epoch=003 worst validation DL-GS gap=0.8280 dB lr=5.00e-05
P=10 seed=20261026 epoch=004 worst validation DL-GS gap=0.8271 dB lr=5.00e-05
P=10 seed=20261026 epoch=005 worst validation DL-GS gap=0.8258 dB lr=5.00e-05
P=10 seed=20261026 epoch=006 worst validation DL-GS gap=0.8247 dB lr=5.00e-05
P=10 seed=20261026 epoch=007 worst validation DL-GS gap=0.8233 dB lr=5.00e-05
P=10 seed=20261026 epoch=008 worst validation DL-GS gap=0.8216 dB lr=5.00e-05
P=10 seed=20261026 epoch=009 worst validation DL-GS gap=0.8211 dB lr=5.00e-05
P=10 seed=20261026 epoch=010 worst validation DL-GS gap=0.8198 dB lr=5.00e-05
P=10 seed=20261026 epoch=011 worst validation DL-GS gap=0.8184 dB lr=5.00e-05
P=10 seed=20261026 epoch=012 worst validation DL-GS gap=0.8173 dB lr=5.00e-05
P=10 seed=20261026 epoch=013 worst va

In [ ]:
import csv
import json
import os

result_dir = '/content/gdrive/MyDrive/unfolded_dl_v8_continue_p10_results'
summary_path = f'{result_dir}/summary_unfolded_v8_p10_continuation.json'

with open(summary_path, encoding='utf-8') as f:
    results = json.load(f)

snrs = [-5, 0, 5, 10, 15, 20, 25, 30]
rows = []

for snr in snrs:
    key = str(snr)
    gaps = [r['test_dl_minus_gs_db_by_snr'][key] for r in results]
    dl_values = [r['test_metrics']['nmse_db_by_snr'][key] for r in results]

    row = {
        'SNR_dB': snr,
        'V8_mean_DL_dB': sum(dl_values) / len(dl_values),
        'V8_mean_DL_minus_GS_dB': sum(gaps) / len(gaps),
        'Seeds_beating_GS': sum(gap < 0 for gap in gaps),
    }
    rows.append(row)

    print(
        f"{snr:>3} dB | mean DL {row['V8_mean_DL_dB']:.3f} dB | "
        f"DL-GS {row['V8_mean_DL_minus_GS_dB']:+.3f} dB | "
        f"beats GS {row['Seeds_beating_GS']}/3 seeds"
    )

csv_path = f'{result_dir}/v8_test_mean_by_snr.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print('Summary saved to:', csv_path)

 -5 dB | mean DL -1.413 dB | DL-GS -9.281 dB | beats GS 3/3 seeds
  0 dB | mean DL -2.906 dB | DL-GS -5.966 dB | beats GS 3/3 seeds
  5 dB | mean DL -5.304 dB | DL-GS -3.185 dB | beats GS 3/3 seeds
 10 dB | mean DL -8.431 dB | DL-GS -1.145 dB | beats GS 3/3 seeds
 15 dB | mean DL -12.792 dB | DL-GS -0.633 dB | beats GS 3/3 seeds
 20 dB | mean DL -17.285 dB | DL-GS -0.101 dB | beats GS 3/3 seeds
 25 dB | mean DL -22.191 dB | DL-GS +0.044 dB | beats GS 0/3 seeds
 30 dB | mean DL -26.764 dB | DL-GS +0.590 dB | beats GS 0/3 seeds
Summary saved to: /content/gdrive/MyDrive/unfolded_dl_v8_continue_p10_results/v8_test_mean_by_snr.csv
